# Python + RDBMS + SQL Training  
## Notebook 1 of 3: RDBMS Concepts, Schema Design, Keys and Basic SQL

### Duration
This notebook is designed for approximately **4 hours** of hands-on learning.

### What you will learn
- What an RDBMS is
- How tables, rows, columns and schema work
- How to design relationships between tables
- Primary key, foreign key, unique key and constraints
- Basic SQL commands: `CREATE`, `INSERT`, `SELECT`, `UPDATE`, `DELETE`
- How to run SQL using Python and SQLite in Google Colab


## 1. What is an RDBMS?

RDBMS stands for **Relational Database Management System**.

It stores data in tables. A table contains rows and columns.

Example:

| student_id | student_name | city |
|---|---|---|
| 1 | Rahul Kumar | Patna |
| 2 | Priya Singh | Kolkata |

In an RDBMS, tables can be connected with each other using keys.

Examples of RDBMS:
- SQLite
- MySQL
- PostgreSQL
- Oracle
- SQL Server
- SAP HANA Cloud


## 2. Why Do We Use Databases?

Without a database, data is often stored in Excel or files. This creates problems:

- Duplicate data
- Manual errors
- No strong relationship between records
- Difficult reporting
- Difficult access control
- Difficult multi-user update

A relational database solves these issues by storing data in a structured, connected and queryable format.


## 3. DBMS vs RDBMS

| Area | DBMS | RDBMS |
|---|---|---|
| Storage style | Files or records | Tables |
| Relationship support | Limited | Strong |
| Keys | May not be strict | Primary key and foreign key |
| Query language | Varies | SQL |
| Examples | File-based DBMS | SQLite, MySQL, PostgreSQL |

An RDBMS is the most common database model used in business applications.


## 4. Core Concepts

| Concept | Meaning |
|---|---|
| Table | Stores data in rows and columns |
| Row | One record |
| Column | One field |
| Schema | Structure of tables and relationships |
| Primary Key | Unique identifier for each row |
| Foreign Key | Connects one table to another table |
| Constraint | Rule applied to data |
| Relationship | Connection between entities |

Example: one student can enroll in many courses. This relationship is represented using tables and keys.


## 5. Case Study: Learning Academy Database

We will create a database for a training academy.

### Tables
1. `departments`
2. `students`
3. `instructors`
4. `courses`
5. `enrollments`
6. `payments`

### Relationships
- One department has many instructors.
- One department has many courses.
- One instructor teaches many courses.
- One student can enroll in many courses.
- One course can have many students.
- One enrollment can have one payment.


## 6. Relationship Diagram

```text
departments 1 ──── many instructors
departments 1 ──── many courses
instructors 1 ──── many courses
students 1 ──── many enrollments
courses 1 ──── many enrollments
enrollments 1 ──── 1 payments
```

The `enrollments` table is important because it connects students and courses.


## 7. SQL Data Types

| Type | Meaning | Example |
|---|---|---|
| INTEGER | Whole number | 101 |
| TEXT | Text value | Rahul |
| REAL | Decimal number | 4999.50 |
| DATE | Date value | 2026-02-01 |

SQLite is flexible with data types, but enterprise databases like PostgreSQL, Oracle and SAP HANA are stricter.


## 8. Keys and Constraints

### Primary Key
Uniquely identifies a row.

### Foreign Key
Connects one table with another.

### Unique Key
Prevents duplicate values.

### Composite Key
A key made from more than one column.

### Check Constraint
Restricts allowed values.

Example:
```sql
CHECK (status IN ('Active', 'Completed', 'Cancelled'))
```


## 9. Set Up SQLite in Python


In [21]:
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

def connect(db_name):
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def exec_script(conn, script):
    conn.executescript(script)
    conn.commit()
    print("Script executed successfully.")

def execute(conn, sql, params=None):
    if params is None:
        params = []
    cur = conn.execute(sql, params)
    conn.commit()
    print(f"Rows affected: {cur.rowcount}")
    return cur

def query(conn, sql, params=None):
    if params is None:
        params = []
    return pd.read_sql_query(sql, conn, params=params)

def tables(conn):
    return query(conn, """
    SELECT name
    FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
    """)

def show_schema(conn, table_name):
    print(f"\nSchema: {table_name}")
    display(query(conn, f"PRAGMA table_info({table_name});"))
    fk = query(conn, f"PRAGMA foreign_key_list({table_name});")
    if not fk.empty:
        print("Foreign Keys:")
        display(fk)

DB_NAME = "rdbms_notebook_01.db"
path = Path(DB_NAME)
if path.exists():
    path.unlink()

conn = connect(DB_NAME)
print("Connected to:", DB_NAME)


Connected to: rdbms_notebook_01.db


## 10. Create Tables

The following SQL script creates all tables with primary keys, foreign keys, unique constraints and check constraints.


In [22]:
schema_sql = """
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE
);

CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    city TEXT,
    registration_date DATE NOT NULL
);

CREATE TABLE instructors (
    instructor_id INTEGER PRIMARY KEY,
    instructor_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    department_id INTEGER NOT NULL,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    department_id INTEGER NOT NULL,
    instructor_id INTEGER NOT NULL,
    fee REAL NOT NULL CHECK (fee >= 0),
    level TEXT NOT NULL CHECK (level IN ('Beginner', 'Intermediate', 'Advanced')),
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)
);

CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrollment_date DATE NOT NULL,
    status TEXT NOT NULL DEFAULT 'Active' CHECK (status IN ('Active', 'Completed', 'Cancelled')),
    UNIQUE(student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    enrollment_id INTEGER NOT NULL UNIQUE,
    amount REAL NOT NULL CHECK (amount >= 0),
    payment_date DATE NOT NULL,
    payment_status TEXT NOT NULL CHECK (payment_status IN ('Paid', 'Pending', 'Refunded')),
    FOREIGN KEY (enrollment_id) REFERENCES enrollments(enrollment_id)
);
"""
exec_script(conn, schema_sql)
tables(conn)


Script executed successfully.


,name
0,courses
1,departments
2,enrollments
3,instructors
4,payments
5,students


## 11. Inspect Table Schema

`PRAGMA table_info(table_name)` shows column details.

`PRAGMA foreign_key_list(table_name)` shows relationships.


In [23]:
for table_name in tables(conn)["name"]:
    show_schema(conn, table_name)



Schema: courses


,cid,name,type,notnull,dflt_value,pk
0,0,course_id,INTEGER,0,None,1
1,1,course_name,TEXT,1,None,0
2,2,department_id,INTEGER,1,None,0
3,3,instructor_id,INTEGER,1,None,0
4,4,fee,REAL,1,None,0
5,5,level,TEXT,1,None,0


Foreign Keys:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,instructors,instructor_id,instructor_id,NO ACTION,NO ACTION,NONE
1,1,0,departments,department_id,department_id,NO ACTION,NO ACTION,NONE



Schema: departments


...

,cid,name,type,notnull,dflt_value,pk
0,0,payment_id,INTEGER,0,None,1
1,1,enrollment_id,INTEGER,1,None,0
2,2,amount,REAL,1,None,0
3,3,payment_date,DATE,1,None,0
4,4,payment_status,TEXT,1,None,0


Foreign Keys:


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,enrollments,enrollment_id,enrollment_id,NO ACTION,NO ACTION,NONE



Schema: students


,cid,name,type,notnull,dflt_value,pk
0,0,student_id,INTEGER,0,None,1
1,1,student_name,TEXT,1,None,0
2,2,email,TEXT,1,None,0
3,3,city,TEXT,0,None,0
4,4,registration_date,DATE,1,None,0


## 12. Insert Sample Data

Parent tables must be inserted first.

Correct order:
1. departments
2. students
3. instructors
4. courses
5. enrollments
6. payments


In [24]:
data_sql = """
INSERT INTO departments VALUES
(1, 'Data Science'),
(2, 'Software Engineering'),
(3, 'Business Analytics');

INSERT INTO students VALUES
(1, 'Rahul Kumar', 'rahul.kumar@example.com', 'Patna', '2026-01-05'),
(2, 'Priya Singh', 'priya.singh@example.com', 'Kolkata', '2026-01-06'),
(3, 'Amit Raj', 'amit.raj@example.com', 'Delhi', '2026-01-07'),
(4, 'Sneha Verma', 'sneha.verma@example.com', 'Patna', '2026-01-10'),
(5, 'Aditya Sharma', 'aditya.sharma@example.com', 'Mumbai', '2026-01-12'),
(6, 'Neha Gupta', 'neha.gupta@example.com', 'Ranchi', '2026-01-13');

INSERT INTO instructors VALUES
(1, 'Dr. Meera Iyer', 'meera.iyer@example.com', 1),
(2, 'Arjun Sen', 'arjun.sen@example.com', 2),
(3, 'Kavita Rao', 'kavita.rao@example.com', 3);

INSERT INTO courses VALUES
(101, 'Python for Beginners', 2, 2, 4999, 'Beginner'),
(102, 'SQL and RDBMS Masterclass', 1, 1, 6999, 'Beginner'),
(103, 'Machine Learning Basics', 1, 1, 11999, 'Intermediate'),
(104, 'Business Dashboarding', 3, 3, 8999, 'Intermediate'),
(105, 'Advanced Data Engineering', 1, 1, 15999, 'Advanced');

INSERT INTO enrollments VALUES
(1001, 1, 101, '2026-02-01', 'Active'),
(1002, 1, 102, '2026-02-03', 'Completed'),
(1003, 2, 102, '2026-02-04', 'Active'),
(1004, 3, 103, '2026-02-05', 'Active'),
(1005, 4, 104, '2026-02-07', 'Cancelled'),
(1006, 5, 105, '2026-02-08', 'Active'),
(1007, 6, 101, '2026-02-08', 'Completed'),
(1008, 2, 104, '2026-02-09', 'Active');

INSERT INTO payments VALUES
(501, 1001, 4999, '2026-02-01', 'Paid'),
(502, 1002, 6999, '2026-02-03', 'Paid'),
(503, 1003, 6999, '2026-02-04', 'Pending'),
(504, 1004, 11999, '2026-02-05', 'Paid'),
(505, 1005, 0, '2026-02-07', 'Refunded'),
(506, 1006, 15999, '2026-02-08', 'Paid'),
(507, 1007, 4999, '2026-02-08', 'Paid'),
(508, 1008, 8999, '2026-02-09', 'Pending');
"""
exec_script(conn, data_sql)


Script executed successfully.


## 13. Basic SELECT Query

`SELECT` is used to fetch data from a table.


In [25]:
query(conn, "SELECT * FROM students;")


,student_id,student_name,email,city,registration_date
0,1,Rahul Kumar,rahul.kumar@example.com,Patna,2026-01-05
1,2,Priya Singh,priya.singh@example.com,Kolkata,2026-01-06
2,3,Amit Raj,amit.raj@example.com,Delhi,2026-01-07
3,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10
4,5,Aditya Sharma,aditya.sharma@example.com,Mumbai,2026-01-12
5,6,Neha Gupta,neha.gupta@example.com,Ranchi,2026-01-13


## 14. Select Specific Columns

In real projects, avoid `SELECT *` when you need only selected columns.


In [26]:
query(conn, """
SELECT student_id, student_name, city
FROM students;
""")


,student_id,student_name,city
0,1,Rahul Kumar,Patna
1,2,Priya Singh,Kolkata
2,3,Amit Raj,Delhi
3,4,Sneha Verma,Patna
4,5,Aditya Sharma,Mumbai
5,6,Neha Gupta,Ranchi


## 15. WHERE Clause

`WHERE` filters rows.


In [27]:
query(conn, """
SELECT student_id, student_name, city
FROM students
WHERE city = 'Patna';
""")


,student_id,student_name,city
0,1,Rahul Kumar,Patna
1,4,Sneha Verma,Patna


## 16. AND, OR and NOT

Use:
- `AND` when all conditions must be true
- `OR` when at least one condition must be true
- `NOT` to reverse a condition


In [28]:
query(conn, """
SELECT course_id, course_name, fee, level
FROM courses
WHERE fee > 7000 AND level = 'Intermediate';
""")


,course_id,course_name,fee,level
0,103,Machine Learning Basics,11999.0,Intermediate
1,104,Business Dashboarding,8999.0,Intermediate


In [29]:
query(conn, """
SELECT student_name, city
FROM students
WHERE city = 'Patna' OR city = 'Kolkata';
""")


,student_name,city
0,Rahul Kumar,Patna
1,Priya Singh,Kolkata
2,Sneha Verma,Patna


In [30]:
query(conn, """
SELECT student_name, city
FROM students
WHERE NOT city = 'Patna';
""")


,student_name,city
0,Priya Singh,Kolkata
1,Amit Raj,Delhi
2,Aditya Sharma,Mumbai
3,Neha Gupta,Ranchi


## 17. ORDER BY and LIMIT

`ORDER BY` sorts records.

`LIMIT` restricts output rows.


In [31]:
query(conn, """
SELECT course_name, fee
FROM courses
ORDER BY fee DESC
LIMIT 3;
""")


,course_name,fee
0,Advanced Data Engineering,15999.0
1,Machine Learning Basics,11999.0
2,Business Dashboarding,8999.0


## 18. DISTINCT

`DISTINCT` removes duplicates.


In [32]:
query(conn, """
SELECT DISTINCT city
FROM students
ORDER BY city;
""")


,city
0,Delhi
1,Kolkata
2,Mumbai
3,Patna
4,Ranchi


## 19. INSERT Using Python Parameters

Parameterized queries use placeholders like `?`.

This is safer than joining strings manually.


In [33]:
new_student = (7, "Rohan Das", "rohan.das@example.com", "Pune", "2026-02-15")

execute(conn, """
INSERT INTO students (student_id, student_name, email, city, registration_date)
VALUES (?, ?, ?, ?, ?);
""", new_student)

query(conn, "SELECT * FROM students WHERE student_id = 7;")


Rows affected: 1


,student_id,student_name,email,city,registration_date
0,7,Rohan Das,rohan.das@example.com,Pune,2026-02-15


## 20. UPDATE Query


In [34]:
execute(conn, """
UPDATE students
SET city = ?
WHERE student_id = ?;
""", ("Bengaluru", 7))

query(conn, "SELECT * FROM students WHERE student_id = 7;")


Rows affected: 1


,student_id,student_name,email,city,registration_date
0,7,Rohan Das,rohan.das@example.com,Bengaluru,2026-02-15


## 21. DELETE Query

Always use a `WHERE` condition while deleting specific rows.


In [35]:
execute(conn, """
DELETE FROM students
WHERE student_id = ?;
""", (7,))

query(conn, "SELECT * FROM students ORDER BY student_id;")


Rows affected: 1


,student_id,student_name,email,city,registration_date
0,1,Rahul Kumar,rahul.kumar@example.com,Patna,2026-01-05
1,2,Priya Singh,priya.singh@example.com,Kolkata,2026-01-06
2,3,Amit Raj,amit.raj@example.com,Delhi,2026-01-07
3,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10
4,5,Aditya Sharma,aditya.sharma@example.com,Mumbai,2026-01-12
5,6,Neha Gupta,neha.gupta@example.com,Ranchi,2026-01-13


## 22. Constraint Demo: Duplicate Email

The following insert fails because email is marked `UNIQUE`.


In [36]:
try:
    execute(conn, """
    INSERT INTO students (student_id, student_name, email, city, registration_date)
    VALUES (?, ?, ?, ?, ?);
    """, (8, "Duplicate User", "rahul.kumar@example.com", "Patna", "2026-03-01"))
except Exception as e:
    print("Error:", e)


Error: UNIQUE constraint failed: students.email


## 23. Constraint Demo: Invalid Foreign Key

The following insert fails because student ID `999` does not exist.


In [37]:
try:
    execute(conn, """
    INSERT INTO enrollments (enrollment_id, student_id, course_id, enrollment_date, status)
    VALUES (?, ?, ?, ?, ?);
    """, (2001, 999, 101, "2026-03-01", "Active"))
except Exception as e:
    print("Error:", e)


Error: FOREIGN KEY constraint failed


## 24. Practice Exercise

Write SQL queries for the following:

1. Show all courses.
2. Show all beginner courses.
3. Show all students from Patna.
4. Show the highest fee course.
5. Show all pending payments.
6. Show students registered after `2026-01-08`.
7. Show all courses sorted by fee from low to high.


In [38]:
# Exercise 1
query(conn, """
SELECT * FROM courses;
""")


,course_id,course_name,department_id,instructor_id,fee,level
0,101,Python for Beginners,2,2,4999.0,Beginner
1,102,SQL and RDBMS Masterclass,1,1,6999.0,Beginner
2,103,Machine Learning Basics,1,1,11999.0,Intermediate
3,104,Business Dashboarding,3,3,8999.0,Intermediate
4,105,Advanced Data Engineering,1,1,15999.0,Advanced


In [39]:
# Exercise 2
query(conn, """
SELECT * FROM courses
WHERE level = 'Beginner';
""")


,course_id,course_name,department_id,instructor_id,fee,level
0,101,Python for Beginners,2,2,4999.0,Beginner
1,102,SQL and RDBMS Masterclass,1,1,6999.0,Beginner


In [40]:
# Exercise 3
query(conn, """
SELECT * FROM payments
WHERE payment_status = 'Pending';
""")


,payment_id,enrollment_id,amount,payment_date,payment_status
0,503,1003,6999.0,2026-02-04,Pending
1,508,1008,8999.0,2026-02-09,Pending


## 25. Notebook 1 Summary

You learned:

- What RDBMS means
- How schema design works
- How primary keys and foreign keys connect tables
- How constraints protect data quality
- How to create tables
- How to insert, read, update and delete records
- How Python can execute SQL queries

Next notebook: joins, aggregations, subqueries, CTEs, window functions and transactions.


In [41]:
print('--- 1. All Courses ---')
display(query(conn, 'SELECT * FROM courses;'))

print('\n--- 2. Beginner Courses ---')
display(query(conn, "SELECT * FROM courses WHERE level = 'Beginner';"))

print('\n--- 3. Students from Patna ---')
display(query(conn, "SELECT * FROM students WHERE city = 'Patna';"))

print('\n--- 4. Highest Fee Course ---')
display(query(conn, 'SELECT * FROM courses ORDER BY fee DESC LIMIT 1;'))

print('\n--- 5. Pending Payments ---')
display(query(conn, "SELECT * FROM payments WHERE payment_status = 'Pending';"))

print('\n--- 6. Students Registered After 2026-01-08 ---')
display(query(conn, "SELECT * FROM students WHERE registration_date > '2026-01-08';"))

print('\n--- 7. Courses Sorted by Fee (Low to High) ---')
display(query(conn, 'SELECT * FROM courses ORDER BY fee ASC;'))

--- 1. All Courses ---


,course_id,course_name,department_id,instructor_id,fee,level
0,101,Python for Beginners,2,2,4999.0,Beginner
1,102,SQL and RDBMS Masterclass,1,1,6999.0,Beginner
2,103,Machine Learning Basics,1,1,11999.0,Intermediate
3,104,Business Dashboarding,3,3,8999.0,Intermediate
4,105,Advanced Data Engineering,1,1,15999.0,Advanced



--- 2. Beginner Courses ---


,course_id,course_name,department_id,instructor_id,fee,level
0,101,Python for Beginners,2,2,4999.0,Beginner
1,102,SQL and RDBMS Masterclass,1,1,6999.0,Beginner



--- 3. Students from Patna ---


...

,payment_id,enrollment_id,amount,payment_date,payment_status
0,503,1003,6999.0,2026-02-04,Pending
1,508,1008,8999.0,2026-02-09,Pending



--- 6. Students Registered After 2026-01-08 ---


,student_id,student_name,email,city,registration_date
0,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10
1,5,Aditya Sharma,aditya.sharma@example.com,Mumbai,2026-01-12
2,6,Neha Gupta,neha.gupta@example.com,Ranchi,2026-01-13



--- 7. Courses Sorted by Fee (Low to High) ---


,course_id,course_name,department_id,instructor_id,fee,level
0,101,Python for Beginners,2,2,4999.0,Beginner
1,102,SQL and RDBMS Masterclass,1,1,6999.0,Beginner
2,104,Business Dashboarding,3,3,8999.0,Intermediate
3,103,Machine Learning Basics,1,1,11999.0,Intermediate
4,105,Advanced Data Engineering,1,1,15999.0,Advanced


Hands-on Problem Statement
RDBMS Foundation, Schema Design, Keys and Basic SQL Using Python
Project Title
Student Training Academy Database Management System
1. Problem Overview
A training academy wants to manage its student enrollment process using a relational database.
Currently, the academy stores student, course, instructor, enrollment, and payment details manually in Excel sheets. This creates problems such as duplicate records, difficulty in tracking enrollments, incorrect payment status, and difficulty in generating reports.
Your task is to design and build a small RDBMS-based system using SQLite and Python.
The system should store structured data in multiple related tables and allow users to perform basic database operations such as creating tables, inserting data, reading records, updating records, deleting records, and validating relationships using keys and constraints.
2. Learning Objectives
By completing this hands-on assignment, learners will understand:
What an RDBMS is
How tables, rows, columns, and schema work
How to design a relational database schema
How to use primary keys and foreign keys
How to apply constraints such as NOT NULL, UNIQUE, CHECK, and DEFAULT
How to insert, read, update, and delete records
How to filter and sort data using SQL
How to use Python to connect with SQLite
How to execute SQL queries from Python
How to fetch SQL results into Pandas DataFrames
3. Business Scenario
The training academy offers multiple courses across different departments.
The academy wants to track:
Departments
Students
Instructors
Courses
Student enrollments
Payments
The academy has the following business rules:
Each department can have many instructors.
Each department can offer many courses.
Each instructor belongs to one department.
Each instructor can teach many courses.
Each student can enroll in many courses.
Each course can have many students.
A student should not be enrolled in the same course twice.
Each enrollment can have one payment record.
Payment status should be one of: Paid, Pending, or Refunded.
Enrollment status should be one of: Active, Completed, or Cancelled.
4. Database Tables Required
You need to create the following tables:
Table 1: departments
This table stores department details.
Columns
Column Name	Data Type	Constraint	Description
department_id	INTEGER	PRIMARY KEY	Unique department ID
department_name	TEXT	NOT NULL, UNIQUE	Name of the department
Sample Data
department_id	department_name
1	Data Science
2	Software Engineering
3	Business Analytics
Table 2: students
This table stores student details.
Columns
Column Name	Data Type	Constraint	Description
student_id	INTEGER	PRIMARY KEY	Unique student ID
student_name	TEXT	NOT NULL	Name of the student
email	TEXT	NOT NULL, UNIQUE	Student email
city	TEXT	Optional	City of student
registration_date	DATE	NOT NULL	Registration date
Sample Data
student_id	student_name	email	city	registration_date
1	Rahul Kumar	rahul.kumar@example.com	Patna	2026-01-05
2	Priya Singh	priya.singh@example.com	Kolkata	2026-01-06
3	Amit Raj	amit.raj@example.com	Delhi	2026-01-07
4	Sneha Verma	sneha.verma@example.com	Patna	2026-01-10
5	Aditya Sharma	aditya.sharma@example.com	Mumbai	2026-01-12
Table 3: instructors
This table stores instructor details.
Columns
Column Name	Data Type	Constraint	Description
instructor_id	INTEGER	PRIMARY KEY	Unique instructor ID
instructor_name	TEXT	NOT NULL	Name of instructor
email	TEXT	NOT NULL, UNIQUE	Instructor email
department_id	INTEGER	FOREIGN KEY	Linked department ID
Relationship
department_id should reference departments(department_id).
Sample Data
instructor_id	instructor_name	email	department_id
1	Dr. Meera Iyer	meera.iyer@example.com	1
2	Arjun Sen	arjun.sen@example.com	2
3	Kavita Rao	kavita.rao@example.com	3
Table 4: courses
This table stores course details.
Columns
Column Name	Data Type	Constraint	Description
course_id	INTEGER	PRIMARY KEY	Unique course ID
course_name	TEXT	NOT NULL	Name of course
department_id	INTEGER	FOREIGN KEY	Linked department ID
instructor_id	INTEGER	FOREIGN KEY	Linked instructor ID
fee	REAL	NOT NULL, CHECK fee >= 0	Course fee
level	TEXT	CHECK	Beginner, Intermediate, Advanced
Relationships
department_id should reference departments(department_id)
instructor_id should reference instructors(instructor_id)
Sample Data
course_id	course_name	department_id	instructor_id	fee	level
101	Python for Beginners	2	2	4999	Beginner
102	SQL and RDBMS Masterclass	1	1	6999	Beginner
103	Machine Learning Basics	1	1	11999	Intermediate
104	Business Dashboarding	3	3	8999	Intermediate
105	Advanced Data Engineering	1	1	15999	Advanced
Table 5: enrollments
This table stores student-course enrollment details.
Columns
Column Name	Data Type	Constraint	Description
enrollment_id	INTEGER	PRIMARY KEY	Unique enrollment ID
student_id	INTEGER	FOREIGN KEY	Linked student ID
course_id	INTEGER	FOREIGN KEY	Linked course ID
enrollment_date	DATE	NOT NULL	Date of enrollment
status	TEXT	DEFAULT 'Active', CHECK	Active, Completed, Cancelled
Relationships
student_id should reference students(student_id)
course_id should reference courses(course_id)
Special Rule
A student should not be enrolled in the same course twice.
Use:
UNIQUE(student_id, course_id)
Sample Data
enrollment_id	student_id	course_id	enrollment_date	status
1001	1	101	2026-02-01	Active
1002	1	102	2026-02-03	Completed
1003	2	102	2026-02-04	Active
1004	3	103	2026-02-05	Active
1005	4	104	2026-02-07	Cancelled
Table 6: payments
This table stores payment details for each enrollment.
Columns
Column Name	Data Type	Constraint	Description
payment_id	INTEGER	PRIMARY KEY	Unique payment ID
enrollment_id	INTEGER	FOREIGN KEY, UNIQUE	Linked enrollment ID
amount	REAL	NOT NULL, CHECK amount >= 0	Payment amount
payment_date	DATE	NOT NULL	Payment date
payment_status	TEXT	CHECK	Paid, Pending, Refunded
Relationship
enrollment_id should reference enrollments(enrollment_id).
Sample Data
payment_id	enrollment_id	amount	payment_date	payment_status
501	1001	4999	2026-02-01	Paid
502	1002	6999	2026-02-03	Paid
503	1003	6999	2026-02-04	Pending
504	1004	11999	2026-02-05	Paid
505	1005	0	2026-02-07	Refunded
5. Hands-on Tasks
Task 1: Create SQLite Database Using Python
Create a database named:
training_academy.db
Use Python’s sqlite3 library.
You should also enable foreign key support using:
conn.execute("PRAGMA foreign_keys = ON;")
Task 2: Create Tables
Create all six tables:
departments
students
instructors
courses
enrollments
payments
Your table design must include:
Primary keys
Foreign keys
NOT NULL
UNIQUE
CHECK
DEFAULT
Composite unique constraint on student_id and course_id
Task 3: Insert Sample Data
Insert at least:
3 departments
5 students
3 instructors
5 courses
5 enrollments
5 payments
Follow the correct insertion order so that foreign key constraints do not fail.
Correct order:
departments → students → instructors → courses → enrollments → payments
Task 4: Display All Tables
Write Python code to display data from all tables using Pandas.
Example:
pd.read_sql_query("SELECT * FROM students", conn)
You should display:
All students
All instructors
All courses
All enrollments
All payments
Task 5: Basic SQL Queries
Write SQL queries for the following:
Show all students.
Show only student name, email, and city.
Show all students from Patna.
Show all courses with fee greater than 7000.
Show all beginner-level courses.
Show all students sorted by student name.
Show the top 3 highest fee courses.
Show distinct student cities.
Show all pending payments.
Show all active enrollments.
Task 6: INSERT Operation Using Python
Add a new student using Python parameterized query.
New student:
student_name	email	city	registration_date
Rohan Das	rohan.das@example.com	Pune	2026-02-15
After insertion, display the student record.
Task 7: UPDATE Operation Using Python
Update the city of Rohan Das from Pune to Bengaluru.
After update, display the updated record.
Task 8: DELETE Operation Using Python
Delete the student Rohan Das.
After deletion, display all students and confirm the record has been removed.
Task 9: Constraint Testing
You need to intentionally test the constraints.
Test 1: UNIQUE Constraint
Try inserting a student with an email that already exists.
Expected result:
The database should reject the insert.
Test 2: FOREIGN KEY Constraint
Try inserting an enrollment with a student ID that does not exist.
Expected result:
The database should reject the insert.
Test 3: CHECK Constraint
Try inserting a course with a negative fee.
Expected result:
The database should reject the insert.
Test 4: CHECK Constraint on Status
Try inserting an enrollment status called Started.
Expected result:
The database should reject the insert because only Active, Completed, and Cancelled are allowed.
Task 10: Schema Inspection
Write Python code to inspect table structure.
Use:
PRAGMA table_info(table_name);
Inspect the schema of:
students
courses
enrollments
payments
Also inspect foreign keys using:
PRAGMA foreign_key_list(table_name);
6. Expected SQL Queries
Learners should write and run these queries:
SELECT * FROM students;
SELECT student_name, email, city
FROM students;
SELECT *
FROM students
WHERE city = 'Patna';
SELECT *
FROM courses
WHERE fee > 7000;
SELECT *
FROM courses
WHERE level = 'Beginner';
SELECT *
FROM students
ORDER BY student_name ASC;
SELECT course_name, fee
FROM courses
ORDER BY fee DESC
LIMIT 3;
SELECT DISTINCT city
FROM students;
SELECT *
FROM payments
WHERE payment_status = 'Pending';
SELECT *
FROM enrollments
WHERE status = 'Active';
7. Expected Python Functions
Create reusable Python functions:
def connect_db(db_name):
    conn = sqlite3.connect(db_name)
        conn.execute("PRAGMA foreign_keys = ON;")
            return conn
            def execute_query(conn, sql, params=None):
                if params is None:
                        params = []
                            cursor = conn.execute(sql, params)
                                conn.commit()
                                    return cursor
                                    def read_query(conn, sql, params=None):
                                        if params is None:
                                                params = []
                                                    return pd.read_sql_query(sql, conn, params=params)
                                                    8. Final Output Expected
                                                    At the end of the hands-on assignment, learners should submit:
                                                    SQLite database file: training_academy.db
                                                    Python notebook or Python script
                                                    SQL table creation script
                                                    Sample data insertion script
                                                    Output screenshots or displayed DataFrames
                                                    Answers to all SQL query tasks
                                                    Constraint testing output
                                                    Short explanation of schema and relationships
                                                    9. Evaluation Checklist
                                                    Evaluation Area	Requirement
                                                    RDBMS understanding	Tables and relationships are correctly explained
                                                    Schema design	All six tables are created properly
                                                    Primary keys	Each table has a valid primary key
                                                    Foreign keys	Relationships are correctly implemented
                                                    Constraints	NOT NULL, UNIQUE, CHECK, DEFAULT are used
                                                    Sample data	Correct sample data inserted
                                                    Basic SQL	SELECT, WHERE, ORDER BY, LIMIT, DISTINCT used
                                                    CRUD	INSERT, UPDATE, DELETE performed from Python
                                                    Constraint testing	Invalid records are blocked
                                                    Python integration	SQLite connection and Pandas display are working
                                                    Code quality	Code is clean, readable, and reusable
                                                    10. Mini Viva / Concept Questions
                                                    Answer the following:
                                                    What is an RDBMS?
                                                    What is the difference between a table and a schema?
                                                    What is a primary key?
                                                    What is a foreign key?
                                                    Why do we use constraints?
                                                    Why should email be marked as UNIQUE?
                                                    Why is UNIQUE(student_id, course_id) used in the enrollments table?
                                                    Why should foreign keys be enabled in SQLite?
                                                    What is the purpose of WHERE?
                                                    What is the difference between UPDATE and DELETE?
                                                    Why are parameterized queries safer?
                                                    Why should parent table records be inserted before child table records?
                                                    11. Bonus Task
                                                    Add a new table named course_feedback.
                                                    Columns
                                                    Column Name	Data Type	Constraint
                                                    feedback_id	INTEGER	PRIMARY KEY
                                                    enrollment_id	INTEGER	FOREIGN KEY
                                                    rating	INTEGER	CHECK rating BETWEEN 1 AND 5
                                                    comments	TEXT	Optional
                                                    feedback_date	DATE	NOT NULL
                                                    Required Queries
                                                    Insert 3 feedback records.
                                                    Show all feedback.
                                                    Show feedback with rating greater than or equal to 4.
                                                    Try inserting rating as 6 and check whether the database rejects it.
                                                    12. Final Learning Outcome
                                                    After completing this hands-on problem, learners will be able to design a relational database from a business requirement, create tables with proper keys and constraints, insert and manage data, execute basic SQL queries, test database rules, and connect Python with SQLite for practical database operations.
                                                     
                                                     SELECT * FROM courses; SELECT * FROM courses WHERE level = 'Beginner'; SELECT * FROM students WHERE city = 'Patna'; SELECT * FROM courses ORDER BY fee DESC LIMIT 1; SELECT * FROM payments WHERE payment_status = 'Pending'; SELECT * FROM students WHERE registration_date > '2026-01-08'; SELECT * FROM courses ORDER BY fee ASC;
                                                      
                                                      1.SELECT * FROM courses
                                                      order by fee desc
                                                      limit 1;
                                                      2.SELECT * FROM courses
                                                      WHERE level = 'Beginner';
                                                      3.ELECT * FROM Students
                                                      WHERE city="patna";
                                                      4.SELECT * FROM courses
                                                      order by fee desc
                                                      limit 1;
                                                      5.SELECT * FROM payments
                                                      WHERE payment_status = 'Pending';
                                                      6.SELECT * FROM Students
                                                      WHERE Registration_date>'2026-01-08';
                                                      7.SELECT * FROM Courses
                                                      order by fee ASC;
                                                       
                                                        Exercise 1
                                                        query(conn, """
                                                        SELECT * FROM courses;
                                                        """)
                                                        Exercise 2
                                                        query(conn, """
                                                        SELECT * FROM courses
                                                        WHERE level = 'Beginner';
                                                        """)
                                                        Exercise 3
                                                        query(conn, """
                                                        select * from students where city='Patna';
                                                        """)
                                                        query(conn,"""
                                                        select *
                                                        from Courses
                                                        WHERE fee = (sekect MAX(fee) from Courses);""")